# CAFA5 Medium Graph Training in Notebook

Objective: run a medium-size graph training experiment directly inside this notebook and plot the results.

This notebook does not submit a Savio Slurm job. It does not call `sbatch`, `squeue`, or `scripts/savio_train_full_graph.sh`. It expects the active notebook kernel to already have the graph runtime installed (`torch`, `torch_geometric`, and repo code), and it reads the prepared `graph_cache` artifacts directly.

Default run:
- prepared data root: `/global/scratch/users/$USER/cafa5_outputs/graph_cache`
- split source: `graph_cache/splits`
- medium subset: `4096 / 1024 / 1024` train/val/test IDs, capped by available IDs
- aspect: `MFO`
- epochs: `5`
- metrics: loss, micro-F1, macro-F1, Fmax

To run all three namespaces in the notebook kernel, set `CAFA5_MEDIUM_ASPECTS=BPO CCO MFO` before running the setup cell, or edit `ASPECTS` in that cell.


In [ ]:
# Setup and configuration
from __future__ import annotations

import gc
import json
import os
import sys
import time
from argparse import Namespace
from datetime import datetime
from pathlib import Path

try:
    import pandas as pd
except ImportError:
    pd = None

try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None

from IPython.display import display

VALID_ASPECTS = {'BPO', 'CCO', 'MFO'}


def find_repo_root(start: Path) -> tuple[Path, str]:
    for candidate in (start, *start.parents):
        if (candidate / '.git').exists():
            return candidate, 'cwd_search'
    return start, 'cwd_fallback'


def resolve_repo_root() -> tuple[Path, str]:
    configured = os.environ.get('CAFA5_REPO_ROOT')
    if configured:
        return Path(configured).expanduser().resolve(), 'env:CAFA5_REPO_ROOT'
    return find_repo_root(Path.cwd())


def parse_aspects(value: str) -> list[str]:
    parsed = [part.strip().upper() for part in value.replace(',', ' ').split() if part.strip()]
    invalid = [aspect for aspect in parsed if aspect not in VALID_ASPECTS]
    if invalid:
        raise ValueError(f'Invalid aspects: {invalid}; expected subset of {sorted(VALID_ASPECTS)}')
    if not parsed:
        raise ValueError('At least one aspect is required')
    return parsed


def env_bool(name: str, default: str = '0') -> bool:
    return os.environ.get(name, default).strip().lower() in {'1', 'true', 'yes', 'y'}


REPO_ROOT, REPO_ROOT_SOURCE = resolve_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

SERVER_USER = os.environ.get('USER', 'bensonli')
RUN_ROOT = Path(os.environ.get('CAFA5_RUN_ROOT', f'/global/scratch/users/{SERVER_USER}/cafa5_outputs'))
GRAPH_CACHE_DIR = RUN_ROOT / 'graph_cache'
FULL_SPLIT_DIR = GRAPH_CACHE_DIR / 'splits'

NOTEBOOK_RUN_ID = os.environ.get('CAFA5_MEDIUM_RUN_ID') or os.environ.get('CAFA5_NOTEBOOK_RUN_ID') or globals().get('NOTEBOOK_RUN_ID') or datetime.now().strftime('%Y%m%d_%H%M%S')
FRAMEWORK = os.environ.get('CAFA5_NOTEBOOK_FRAMEWORK', 'pyg')
ASPECTS = parse_aspects(os.environ.get('CAFA5_MEDIUM_ASPECTS', os.environ.get('CAFA5_NOTEBOOK_ASPECTS', 'MFO')))
SPLIT_MODE = os.environ.get('CAFA5_NOTEBOOK_SPLIT_MODE', 'medium').lower()
MIN_TERM_FREQUENCY = int(os.environ.get('CAFA5_MIN_TERM_FREQUENCY', '20'))
SEED = int(os.environ.get('CAFA5_TRAIN_SEED', '2026'))

MEDIUM_TRAIN_LIMIT = int(os.environ.get('CAFA5_MEDIUM_TRAIN_LIMIT', '4096'))
MEDIUM_VAL_LIMIT = int(os.environ.get('CAFA5_MEDIUM_VAL_LIMIT', '1024'))
MEDIUM_TEST_LIMIT = int(os.environ.get('CAFA5_MEDIUM_TEST_LIMIT', '1024'))
EPOCHS = int(os.environ.get('CAFA5_MEDIUM_EPOCHS', os.environ.get('CAFA5_NOTEBOOK_EPOCHS', '5')))
BATCH_SIZE = int(os.environ.get('CAFA5_MEDIUM_BATCH_SIZE', os.environ.get('CAFA5_NOTEBOOK_BATCH_SIZE', '8')))
NUM_WORKERS = int(os.environ.get('CAFA5_MEDIUM_NUM_WORKERS', os.environ.get('CAFA5_NOTEBOOK_NUM_WORKERS', '0')))
HIDDEN_DIM = int(os.environ.get('CAFA5_MEDIUM_HIDDEN_DIM', os.environ.get('CAFA5_NOTEBOOK_HIDDEN_DIM', '128')))
DROPOUT = float(os.environ.get('CAFA5_NOTEBOOK_DROPOUT', '0.2'))
LR = float(os.environ.get('CAFA5_NOTEBOOK_LR', '0.001'))
WEIGHT_DECAY = float(os.environ.get('CAFA5_NOTEBOOK_WEIGHT_DECAY', '0.0001'))
METRIC_THRESHOLD = float(os.environ.get('CAFA5_METRIC_THRESHOLD', '0.5'))
FMAX_THRESHOLD_STEP = float(os.environ.get('CAFA5_FMAX_THRESHOLD_STEP', '0.01'))
PROGRESS_MODE = os.environ.get('CAFA5_MEDIUM_PROGRESS_MODE', os.environ.get('CAFA5_NOTEBOOK_PROGRESS_MODE', 'tqdm'))
PROGRESS_EVERY = int(os.environ.get('CAFA5_MEDIUM_PROGRESS_EVERY', os.environ.get('CAFA5_NOTEBOOK_PROGRESS_EVERY', '25')))
DEVICE = os.environ.get('CAFA5_NOTEBOOK_DEVICE', 'auto')
RUN_TRAINING = env_bool('CAFA5_MEDIUM_RUN_TRAINING', os.environ.get('CAFA5_NOTEBOOK_RUN_TRAINING', '1'))
USE_ESM2 = env_bool('CAFA5_USE_ESM2', '1')
USE_DSSP = env_bool('CAFA5_USE_DSSP', '1')
USE_SASA = env_bool('CAFA5_USE_SASA', '1')

if SPLIT_MODE not in {'medium', 'full'}:
    raise ValueError("SPLIT_MODE must be 'medium' or 'full'")

MEDIUM_SPLIT_DIR = GRAPH_CACHE_DIR / f'splits_medium_notebook_t{MEDIUM_TRAIN_LIMIT}_v{MEDIUM_VAL_LIMIT}_s{MEDIUM_TEST_LIMIT}_mtf{MIN_TERM_FREQUENCY}'
ACTIVE_SPLIT_DIR = MEDIUM_SPLIT_DIR if SPLIT_MODE == 'medium' else FULL_SPLIT_DIR
RUN_NAME = os.environ.get('CAFA5_MEDIUM_RUN_NAME') or os.environ.get('CAFA5_NOTEBOOK_RUN_NAME') or f'notebook_{SPLIT_MODE}_{FRAMEWORK}_mtf{MIN_TERM_FREQUENCY}_{NOTEBOOK_RUN_ID}'
RUN_DIR = GRAPH_CACHE_DIR / 'training_runs' / RUN_NAME

config = {
    'repo_root': str(REPO_ROOT),
    'repo_root_source': REPO_ROOT_SOURCE,
    'run_root': str(RUN_ROOT),
    'graph_cache_dir': str(GRAPH_CACHE_DIR),
    'full_split_dir': str(FULL_SPLIT_DIR),
    'active_split_dir': str(ACTIVE_SPLIT_DIR),
    'split_mode': SPLIT_MODE,
    'run_dir': str(RUN_DIR),
    'framework': FRAMEWORK,
    'aspects': ASPECTS,
    'min_term_frequency': MIN_TERM_FREQUENCY,
    'seed': SEED,
    'limits': {'train': MEDIUM_TRAIN_LIMIT, 'val': MEDIUM_VAL_LIMIT, 'test': MEDIUM_TEST_LIMIT},
    'epochs': EPOCHS,
    'batch_size': BATCH_SIZE,
    'num_workers': NUM_WORKERS,
    'hidden_dim': HIDDEN_DIM,
    'dropout': DROPOUT,
    'lr': LR,
    'weight_decay': WEIGHT_DECAY,
    'metric_threshold': METRIC_THRESHOLD,
    'fmax_threshold_step': FMAX_THRESHOLD_STEP,
    'progress_mode': PROGRESS_MODE,
    'progress_every': PROGRESS_EVERY,
    'device': DEVICE,
    'run_training': RUN_TRAINING,
    'modalities': {'esm2': USE_ESM2, 'dssp': USE_DSSP, 'sasa': USE_SASA},
}
print(json.dumps(config, indent=2))

required_files = [REPO_ROOT / 'train_minimal_graph_model.py', REPO_ROOT / 'cafa_graph_dataloaders.py', REPO_ROOT / 'cafa_graph_dataset.py']
missing_files = [str(path) for path in required_files if not path.exists()]
if missing_files:
    raise FileNotFoundError(f'Missing repo files: {missing_files}')

## Import Graph Runtime

This training happens in the notebook kernel, so the kernel itself must be the graph environment. If this cell fails on `torch_geometric`, switch to the graph-capable kernel/environment before continuing.

In [ ]:
# Import repo training modules and verify runtime.
import torch

import cafa_graph_dataloaders as dataloaders
import cafa_graph_dataset as graphs
import train_minimal_graph_model as training

if FRAMEWORK == 'pyg':
    import torch_geometric
    print('torch_geometric =', torch_geometric.__version__)
elif FRAMEWORK == 'dgl':
    import dgl
    print('dgl =', dgl.__version__)
else:
    raise ValueError(f'Unknown framework: {FRAMEWORK}')

print('python =', sys.executable)
print('torch =', torch.__version__)
print('cuda_available =', torch.cuda.is_available())
if torch.cuda.is_available():
    print('cuda_device_count =', torch.cuda.device_count())
    print('cuda_device_name =', torch.cuda.get_device_name(0))

## Prepare Splits

For `SPLIT_MODE='medium'`, this cell creates a capped split directory from the prepared full split files. For `SPLIT_MODE='full'`, it uses `graph_cache/splits` directly.

In [ ]:
# Build or validate the active split directory.
def read_ids(path: Path) -> list[str]:
    return [line.strip() for line in path.read_text(encoding='utf-8').splitlines() if line.strip()]


def write_lines(path: Path, values: list[str]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(''.join(f'{value}\n' for value in values), encoding='utf-8')
    print(f'[write] {path} ({len(values)} lines)')


if not FULL_SPLIT_DIR.exists():
    raise FileNotFoundError(f'Missing prepared split directory: {FULL_SPLIT_DIR}')
if not (FULL_SPLIT_DIR / 'summary.json').exists():
    raise FileNotFoundError(f'Missing prepared split summary: {FULL_SPLIT_DIR / "summary.json"}')

split_rows = []
if SPLIT_MODE == 'medium':
    limits = {'train': MEDIUM_TRAIN_LIMIT, 'val': MEDIUM_VAL_LIMIT, 'test': MEDIUM_TEST_LIMIT}
    split_summary = {
        'source_split_dir': str(FULL_SPLIT_DIR),
        'root': str(ACTIVE_SPLIT_DIR),
        'split_mode': SPLIT_MODE,
        'run_name': RUN_NAME,
        'min_term_frequency': MIN_TERM_FREQUENCY,
        'seed': SEED,
        'limits': limits,
        'aspects': {},
    }
    for aspect in ASPECTS:
        source_aspect_dir = FULL_SPLIT_DIR / aspect.lower()
        target_aspect_dir = ACTIVE_SPLIT_DIR / aspect.lower()
        if not source_aspect_dir.exists():
            raise FileNotFoundError(f'Missing source split directory for {aspect}: {source_aspect_dir}')
        split_summary['aspects'][aspect] = {'source': str(source_aspect_dir), 'counts': {}, 'source_counts': {}}
        for split_name, limit in limits.items():
            source_path = source_aspect_dir / f'{split_name}.txt'
            ids = read_ids(source_path)
            subset = ids[:min(limit, len(ids))]
            write_lines(target_aspect_dir / f'{split_name}.txt', subset)
            split_summary['aspects'][aspect]['counts'][split_name] = len(subset)
            split_summary['aspects'][aspect]['source_counts'][split_name] = len(ids)
            split_rows.append({'aspect': aspect, 'split': split_name, 'selected': len(subset), 'source_total': len(ids)})
        source_summary_path = source_aspect_dir / 'summary.json'
        if source_summary_path.exists():
            (target_aspect_dir / 'summary.json').write_text(source_summary_path.read_text(encoding='utf-8'), encoding='utf-8')
    ACTIVE_SPLIT_DIR.mkdir(parents=True, exist_ok=True)
    (ACTIVE_SPLIT_DIR / 'summary.json').write_text(json.dumps(split_summary, indent=2), encoding='utf-8')
    print(f'[write] {ACTIVE_SPLIT_DIR / "summary.json"}')
else:
    for aspect in ASPECTS:
        for split_name in ('train', 'val', 'test'):
            path = ACTIVE_SPLIT_DIR / aspect.lower() / f'{split_name}.txt'
            ids = read_ids(path)
            split_rows.append({'aspect': aspect, 'split': split_name, 'selected': len(ids), 'source_total': len(ids)})

if pd is not None:
    display(pd.DataFrame(split_rows))
else:
    print(json.dumps(split_rows, indent=2))

## Training Helpers

The next cells use the shared repo training module, but the loop is run directly here so you can inspect each object and plot results without leaving the notebook.

In [ ]:
# Notebook training helpers.
def make_args(aspect: str, checkpoint_dir: Path) -> Namespace:
    return Namespace(
        root=GRAPH_CACHE_DIR,
        split_dir=ACTIVE_SPLIT_DIR,
        framework=FRAMEWORK,
        aspect=aspect,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        num_workers=NUM_WORKERS,
        hidden_dim=HIDDEN_DIM,
        dropout=DROPOUT,
        lr=LR,
        weight_decay=WEIGHT_DECAY,
        metric_threshold=METRIC_THRESHOLD,
        fmax_threshold_step=FMAX_THRESHOLD_STEP,
        progress_mode=PROGRESS_MODE,
        progress_every=PROGRESS_EVERY,
        device=DEVICE,
        seed=SEED,
        checkpoint_dir=checkpoint_dir,
        min_term_frequency=MIN_TERM_FREQUENCY,
        disable_esm2=not USE_ESM2,
        disable_dssp=not USE_DSSP,
        disable_sasa=not USE_SASA,
    )


def metric_row(record: dict, aspect: str, split_name: str) -> dict:
    metrics = record.get(split_name, {})
    row = {
        'aspect': aspect,
        'split': split_name,
        'epoch': record.get('epoch'),
        'epoch_seconds': record.get('epoch_seconds'),
    }
    for key in (
        'loss',
        'micro_precision',
        'micro_recall',
        'micro_f1',
        'macro_precision',
        'macro_recall',
        'macro_f1',
        'macro_f1_positive_labels',
        'fmax',
        'fmax_threshold',
        'fmax_precision',
        'fmax_recall',
        'graphs',
    ):
        row[key] = metrics.get(key)
    return row


def summarize_epoch_rows(summaries: dict[str, dict]) -> list[dict]:
    rows = []
    for aspect, summary in summaries.items():
        for record in summary.get('history', []):
            for split_name in ('train', 'val', 'test'):
                rows.append(metric_row(record, aspect=aspect, split_name=split_name))
    return rows


def write_results_summary(run_dir: Path, summaries: dict[str, dict]) -> None:
    rows = []
    for aspect, summary in summaries.items():
        history = summary.get('history', [])
        final_epoch = history[-1] if history else {}
        row = {
            'aspect': aspect,
            'device': summary.get('device'),
            'best_val_loss': summary.get('best_val_loss'),
            'best_checkpoint_path': summary.get('best_checkpoint_path'),
            'final_epoch': final_epoch.get('epoch'),
        }
        for split_name in ('train', 'val', 'test'):
            metrics = final_epoch.get(split_name, {})
            for key in ('loss', 'micro_f1', 'macro_f1', 'fmax', 'fmax_threshold', 'graphs'):
                row[f'{split_name}_{key}'] = metrics.get(key)
        rows.append(row)
    run_dir.mkdir(parents=True, exist_ok=True)
    (run_dir / 'results_summary.json').write_text(json.dumps(rows, indent=2), encoding='utf-8')
    if pd is not None:
        pd.DataFrame(rows).to_csv(run_dir / 'results_summary.tsv', sep='\t', index=False)
    else:
        columns = sorted({key for row in rows for key in row})
        with (run_dir / 'results_summary.tsv').open('w', encoding='utf-8') as handle:
            handle.write('\t'.join(columns) + '\n')
            for row in rows:
                handle.write('\t'.join('' if row.get(column) is None else str(row.get(column)) for column in columns) + '\n')

In [ ]:
# Direct notebook training loop.
def unwrap_dataset(dataset):
    # PyG wraps CafaPyGDataset in _PygGraphLevelWrapper; the original dataset holds vocab.
    seen = set()
    while hasattr(dataset, 'dataset') and id(dataset) not in seen:
        if hasattr(dataset, 'vocab'):
            return dataset
        seen.add(id(dataset))
        dataset = dataset.dataset
    return dataset


def loader_dataset(loader):
    return unwrap_dataset(loader.dataset)


def loader_label_count(loader, model=None) -> int | None:
    dataset = loader_dataset(loader)
    vocab = getattr(dataset, 'vocab', None)
    if vocab is not None:
        return len(vocab)
    classifier = getattr(model, 'classifier', None)
    out_features = getattr(classifier, 'out_features', None)
    return int(out_features) if out_features is not None else None


def train_one_aspect(aspect: str) -> dict:
    checkpoint_dir = RUN_DIR / aspect.lower()
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    args = make_args(aspect=aspect, checkpoint_dir=checkpoint_dir)

    training.require_torch()
    training.set_random_seed(args.seed)
    device = training.resolve_device(args.device)
    loaders, model = training.build_training_objects(args)
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=args.lr, weight_decay=args.weight_decay)
    loss_fn = torch.nn.BCEWithLogitsLoss()

    history = []
    best_val_loss = float('inf')
    best_checkpoint_path = checkpoint_dir / 'best.pt'
    summary_path = checkpoint_dir / 'summary.json'

    label_count = loader_label_count(loaders['train'], model)
    print(f'[{aspect}] device={device} train={len(loaders["train"].dataset)} val={len(loaders["val"].dataset)} test={len(loaders["test"].dataset)} labels={label_count}')
    for epoch in range(1, args.epochs + 1):
        epoch_start = time.perf_counter()
        train_metrics = training.run_epoch(
            model=model,
            loader=loaders['train'],
            framework=args.framework,
            device=device,
            optimizer=optimizer,
            loss_fn=loss_fn,
            metric_threshold=args.metric_threshold,
            fmax_threshold_step=args.fmax_threshold_step,
            epoch=epoch,
            split_name=f'{aspect} train',
            progress_mode=args.progress_mode,
            progress_every=args.progress_every,
        )
        val_metrics = training.run_epoch(
            model=model,
            loader=loaders['val'],
            framework=args.framework,
            device=device,
            optimizer=None,
            loss_fn=loss_fn,
            metric_threshold=args.metric_threshold,
            fmax_threshold_step=args.fmax_threshold_step,
            epoch=epoch,
            split_name=f'{aspect} val',
            progress_mode=args.progress_mode,
            progress_every=args.progress_every,
        ) if len(loaders['val'].dataset) > 0 else training.empty_metrics(args.metric_threshold, args.fmax_threshold_step)
        test_metrics = training.run_epoch(
            model=model,
            loader=loaders['test'],
            framework=args.framework,
            device=device,
            optimizer=None,
            loss_fn=loss_fn,
            metric_threshold=args.metric_threshold,
            fmax_threshold_step=args.fmax_threshold_step,
            epoch=epoch,
            split_name=f'{aspect} test',
            progress_mode=args.progress_mode,
            progress_every=args.progress_every,
        ) if len(loaders['test'].dataset) > 0 else training.empty_metrics(args.metric_threshold, args.fmax_threshold_step)

        epoch_seconds = time.perf_counter() - epoch_start
        record = {'epoch': epoch, 'epoch_seconds': epoch_seconds, 'train': train_metrics, 'val': val_metrics, 'test': test_metrics}
        history.append(record)
        print(
            f'[{aspect}] epoch={epoch} '
            f'train_loss={train_metrics["loss"]:.4f} train_micro_f1={train_metrics["micro_f1"]:.4f} train_fmax={train_metrics["fmax"]:.4f} '
            f'val_loss={val_metrics["loss"]:.4f} val_micro_f1={val_metrics["micro_f1"]:.4f} val_fmax={val_metrics["fmax"]:.4f} '
            f'test_loss={test_metrics["loss"]:.4f} test_micro_f1={test_metrics["micro_f1"]:.4f} test_fmax={test_metrics["fmax"]:.4f} '
            f'seconds={epoch_seconds:.2f}'
        )

        if val_metrics['loss'] <= best_val_loss:
            best_val_loss = val_metrics['loss']
            torch.save(
                {
                    'framework': args.framework,
                    'aspect': args.aspect,
                    'epoch': epoch,
                    'model_state': model.state_dict(),
                    'optimizer_state': optimizer.state_dict(),
                    'val_loss': val_metrics['loss'],
                    'val_micro_f1': val_metrics['micro_f1'],
                    'val_macro_f1': val_metrics['macro_f1'],
                    'val_fmax': val_metrics['fmax'],
                    'val_fmax_threshold': val_metrics['fmax_threshold'],
                    'args': vars(args),
                },
                best_checkpoint_path,
            )

        summary = {
            'framework': args.framework,
            'aspect': args.aspect,
            'device': str(device),
            'epochs': args.epochs,
            'batch_size': args.batch_size,
            'hidden_dim': args.hidden_dim,
            'dropout': args.dropout,
            'lr': args.lr,
            'weight_decay': args.weight_decay,
            'metric_threshold': args.metric_threshold,
            'fmax_threshold_step': args.fmax_threshold_step,
            'progress_mode': args.progress_mode,
            'progress_every': args.progress_every,
            'best_val_loss': best_val_loss,
            'best_checkpoint_path': str(best_checkpoint_path.resolve()),
            'history': history,
        }
        summary_path.write_text(json.dumps(summary, indent=2), encoding='utf-8')

    del model, optimizer, loss_fn, loaders
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()
    return summary

## Run Training

This cell performs the actual training in the notebook kernel. Set `CAFA5_NOTEBOOK_RUN_TRAINING=0` if you only want to build splits and inspect configuration.

In [ ]:
# Run training for the configured aspects.
RUN_DIR.mkdir(parents=True, exist_ok=True)
(RUN_DIR / 'notebook_config.json').write_text(json.dumps(config, indent=2), encoding='utf-8')

summaries: dict[str, dict] = {}
if RUN_TRAINING:
    for aspect in ASPECTS:
        summaries[aspect] = train_one_aspect(aspect)
    write_results_summary(RUN_DIR, summaries)
    print('run_dir =', RUN_DIR)
else:
    print('[skip] RUN_TRAINING is false')
    for aspect in ASPECTS:
        summary_path = RUN_DIR / aspect.lower() / 'summary.json'
        if summary_path.exists():
            summaries[aspect] = json.loads(summary_path.read_text(encoding='utf-8'))
            print(f'[load] {summary_path}')

## Results Table

This cell loads the run summaries and turns per-epoch metrics into a table.

In [ ]:
# Load and display per-epoch results.
if not summaries:
    for aspect in ASPECTS:
        summary_path = RUN_DIR / aspect.lower() / 'summary.json'
        if summary_path.exists():
            summaries[aspect] = json.loads(summary_path.read_text(encoding='utf-8'))
if not summaries:
    raise FileNotFoundError(f'No training summaries found under {RUN_DIR}')

epoch_rows = summarize_epoch_rows(summaries)
if pd is not None:
    epoch_df = pd.DataFrame(epoch_rows)
    display(epoch_df.head(20))
    display(epoch_df.sort_values(['aspect', 'split', 'epoch']).groupby(['aspect', 'split'], as_index=False).tail(1))
else:
    print(json.dumps(epoch_rows[:20], indent=2))

## Plots

These plots use the unweighted metrics emitted by the notebook training loop. CAFA IA-weighted `wFmax` still requires prediction TSV export and `cafaeval`.

In [ ]:
# Plot loss, micro-F1, macro-F1, and Fmax over epochs.
if pd is None or plt is None:
    raise ImportError('pandas and matplotlib are required for plotting')
if epoch_df.empty:
    raise ValueError('epoch_df is empty')

metrics_to_plot = ['loss', 'micro_f1', 'macro_f1', 'fmax']
fig, axes = plt.subplots(len(metrics_to_plot), len(ASPECTS), figsize=(5 * len(ASPECTS), 3.2 * len(metrics_to_plot)), squeeze=False)
for col_index, aspect in enumerate(ASPECTS):
    aspect_df = epoch_df[epoch_df['aspect'] == aspect]
    for row_index, metric in enumerate(metrics_to_plot):
        ax = axes[row_index][col_index]
        for split_name in ('train', 'val', 'test'):
            split_df = aspect_df[aspect_df['split'] == split_name].sort_values('epoch')
            if split_df.empty:
                continue
            ax.plot(split_df['epoch'], split_df[metric], marker='o', label=split_name)
        ax.set_title(f'{aspect} {metric}')
        ax.set_xlabel('epoch')
        ax.set_ylabel(metric)
        ax.grid(True, alpha=0.3)
        if row_index == 0 and col_index == 0:
            ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Final val/test comparison by aspect.
if pd is None or plt is None:
    raise ImportError('pandas and matplotlib are required for plotting')
final_df = epoch_df.sort_values('epoch').groupby(['aspect', 'split'], as_index=False).tail(1)
plot_df = final_df[final_df['split'].isin(['val', 'test'])][['aspect', 'split', 'micro_f1', 'macro_f1', 'fmax']]
display(plot_df)

fig, axes = plt.subplots(1, 3, figsize=(15, 4), squeeze=False)
for ax, metric in zip(axes[0], ['micro_f1', 'macro_f1', 'fmax']):
    pivot = plot_df.pivot(index='aspect', columns='split', values=metric).reindex(ASPECTS)
    pivot.plot(kind='bar', ax=ax)
    ax.set_title(f'Final {metric}')
    ax.set_xlabel('aspect')
    ax.set_ylabel(metric)
    ax.set_ylim(0, 1)
    ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## Notes

Record observations after the run:

- Which aspect finished successfully in the notebook kernel?
- Is validation Fmax improving across epochs?
- Is train/val loss diverging?
- Is the medium subset stable enough to move to `SPLIT_MODE='full'`?
- Next step for CAFA-style scoring: export prediction TSV and run `cafaeval` with `IA.txt`.
